In [ ]:
# packages
using Markdown
using InteractiveUtils
using NonlinearSolve
using StaticArrays
# files
using PKAssetPrices
import PKAssetPrices.Static: Parametrization, Model
using GLMakie

## Trying to develop a comparison

### Show parameters

In [21]:
Static.AssetPK.model

nothing

                                                         Model
┌────────────────────────────────┬─────────────────────────────────────────────┬──────────────────────────────────────┐
│                      Variables │                                  Parameters │                            Equations │
│                     Amount: 17 │                                  Amount: 21 │                           Amount: 17 │
├────────────────────────────────┼─────────────────────────────────────────────┼──────────────────────────────────────┤
│                      Y: Output │       d0: autonomous credit financed demand │                      Y == ND + c * D │
│   ND: Non debt-financed demand │                         b: consumption rate │                          ND == b * Y │
│        D: debt-financed demand │           i1: inflation infuced policy rate │                     D == d0 - d1 * r │
│                 i: policy rate │                        W0: autonomous wages │                 

In [22]:
Static.AssetPKSDcompare.model

nothing

                                                         Model
┌────────────────────────────────┬─────────────────────────────────────────────┬──────────────────────────────────────┐
│                      Variables │                                  Parameters │                            Equations │
│                     Amount: 17 │                                  Amount: 22 │                           Amount: 17 │
├────────────────────────────────┼─────────────────────────────────────────────┼──────────────────────────────────────┤
│                      Y: Output │       d0: autonomous credit financed demand │                      Y == ND + c * D │
│   ND: Non debt-financed demand │                         b: consumption rate │                          ND == b * Y │
│        D: debt-financed demand │           i1: inflation infuced policy rate │                     D == d0 - d1 * r │
│                 i: policy rate │                        W0: autonomous wages │                 

In [23]:
scen1a = Static.@scenario Static.AssetPKSDcompare begin
    s2 = 0.80
end

Parametrization
───────────────
Model:      17 vars, 22 params, 17 eqs, 4 curves, 3 sheets
Params:     22 set / 22 declared
u0 length:  17

Parameters
──────────
d0 = 5.0
b  = 0.5
i1 = 0.05
W0 = 2.0
i0 = 0.01
n  = 0.15
s0 = 0.5
c  = 0.8
h  = 0.8
Nᶠ = 6.0
d1 = 8.0
p1 = 1.0
s1 = 1.0
γ0 = 0.0
AQ = 6.0
α₀ = 0.1
s2 = 0.8
k  = 0.3
a  = 0.8
m  = 0.15
γ  = 0.5
gₐ = 0.03


### Show results

In [25]:
sol1 = solve_model(Static.AssetPK)
sol2 = solve_model(Static.AssetPKSDcompare)
sol_scen1a = solve_model(scen1a);

In [26]:
Static.build_table_from_solutions([sol1, sol2, sol_scen1a])

Variable,Scenario 1: Baseline,Scenario 2: Exogenous Alpha,Scenario 3: Central Bank Sense
Y,6.5660179146289535,6.566017914628954,6.5660179146289535
ND,3.2830089573144767,3.283008957314477,3.2830089573144767
D,4.103761196643096,4.103761196643096,4.103761196643096
i,0.09741726123444606,0.09741726123444606,0.09741726123444606
r,0.11202985041961297,0.11202985041961297,0.11202985041961297
P,1.748345224688921,1.7483452246889213,1.7483452246889213
dL,3.670979106894864,-1.028926296464005,3.6572400306241075
dM,3.670979106894864,-1.028926296464005,3.6572400306241075
dR,1.1012937320684593,-0.30867788893920145,1.0971720091872323
W,1.9003752442270883,1.9003752442270885,1.9003752442270885


### Get Balance Sheets

In [ ]:
display(html"<h3>Baseline</h3>")
display(sol1.sheets)
display(html"<h3>with s2=1</h3>")
display(sol2.sheets)
display(html"<h3>with s2=0.8</h3>")
display(sol_scen1a.sheets)

HTML{String}("<h3>Baseline</h3>")

Assets,Value,Liabilities,Value
Deposits,3.67,Loans,3.67
Total,3.67,Total,3.67
Assets,Value,Liabilities,Value
Loans,3.67,Deposits,3.67
Reserves,1.1,Central Bank Credit,1.1
Total,4.77,Total,4.77
Assets,Value,Liabilities,Value
Central Bank Credit,1.1,Reserves,1.1
Total,1.1,Total,1.1


HTML{String}("<h3>with s2=1</h3>")

Assets,Value,Liabilities,Value
Deposits,-1.03,Loans,-1.03
Total,-1.03,Total,-1.03
Assets,Value,Liabilities,Value
Loans,-1.03,Deposits,-1.03
Reserves,-0.31,Central Bank Credit,-0.31
Total,-1.34,Total,-1.34
Assets,Value,Liabilities,Value
Central Bank Credit,-0.31,Reserves,-0.31
Total,-0.31,Total,-0.31


HTML{String}("<h3>with s2=0.8</h3>")

Assets,Value,Liabilities,Value
Deposits,3.66,Loans,3.66
Total,3.66,Total,3.66
Assets,Value,Liabilities,Value
Loans,3.66,Deposits,3.66
Reserves,1.1,Central Bank Credit,1.1
Total,4.75,Total,4.75
Assets,Value,Liabilities,Value
Central Bank Credit,1.1,Reserves,1.1
Total,1.1,Total,1.1


### Show Curves

In [ ]:
IS(r) = begin 
    sol_c1 = solve_model(Static.AssetPK).variables
    sol[:r] = r
    Static.eval_curve(Static.AssetPK, sol_c1).IS
end
IR(Y) = begin 
    sol_c1 = solve_model(Static.AssetPK).variables
    sol_c1[:Y] = Y
    Static.eval_curve(Static.AssetPK, sol_c1).IR
end
f = Figure()
a = Axis(f[1, 1], title = "IS curve", xlabel = "r", ylabel = "Y")
b = Axis(f[1, 2], title = "IR curve", xlabel = "Y", ylabel = "r")
lines!(a, [IS(x) for x in 0:0.01:1])
lines!(b, [IR(x) for x in 0:10:100])

UndefVarError: UndefVarError: `Figure` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [37]:
display(f)

UndefVarError: UndefVarError: `f` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

#### Check alternative Specification

In [31]:
g = x -> solve_model(Static.@scenario Static.AssetPKSDcompare begin
    s2 = x
end)



#56 (generic function with 1 method)

In [32]:
Plots.plot(LinRange(0.5,0.9,100),map(g.(LinRange(0.5,0.9,100))) do x 
   x.r
end)

UndefVarError: UndefVarError: `Plots` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [33]:
using MacroTools

ex2 = eval.([MacroTools.postwalk(ex.rhs) do x
    if !(x isa Symbol)
        return x
    end
    if haskey(sol_scen1a.variables, x)
        return sol_scen1a.variables[x]
    elseif haskey(sol_scen1a.model.params, x)
        return sol_scen1a.model.params[x]
    else
        return x
    end
end for ex in sol_scen1a.model.model.equations]) -eval.([MacroTools.postwalk(ex.lhs) do x
    if !(x isa Symbol)
        return x
    end
    if haskey(sol_scen1a.variables, x)
        return sol_scen1a.variables[x]
    elseif haskey(sol_scen1a.model.params, x)
        return sol_scen1a.model.params[x]
    else
        return x
    end
end for ex in sol_scen1a.model.model.equations])




17-element Vector{Float64}:
  8.881784197001252e-16
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
 -2.220446049250313e-16
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
 -1.1102230246251565e-16
  0.0

In [34]:
sol_scen1a.variables

Dict{Symbol, Float64} with 17 entries:
  :α  => 0.0991413
  :AS => 0.774848
  :N  => 5.25281
  :ND => 3.28301
  :SD => 0.374231
  :D  => 4.10376
  :AD => 0.761541
  :Y  => 6.56602
  :AP => 0.982826
  :r  => 0.11203
  :dL => 3.65724
  :P  => 1.74835
  :U  => 0.124531
  :W  => 1.90038
  :dM => 3.65724
  :i  => 0.0974173
  :dR => 1.09717